In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import importlib
import sys
import pandas as pd

In [ ]:
sys.path.append("../src")

import eda_utils as eda
import feature_engineering as fe
import modeling as mod
import preprocessing as prep
import constants as const
import experiment_utils as exp
import visualization as visual

importlib.reload(eda)
importlib.reload(fe)
importlib.reload(mod)
importlib.reload(prep)
importlib.reload(const)
importlib.reload(exp)
importlib.reload(visual)

<h1 style="
    background-color: #d0ebff;
    color: #1a1a1a;
    display: inline-block;
    padding: 10px 18px;
    border-radius: 10px;
    font-size: 32px;
">
Feature Engineering Experiments
</h1>

Este notebook funciona como módulo de experimentación para decidir qué features conviene llevar al pipeline final de feature engenieering.

- En este notebook se prueban distintas combinaciones de features, se entrenan modelos comparables y se elige la configuración con mejor desempeño en validation.
- En [feature_engineering.ipynb](feature_engineering.ipynb) se aplica únicamente la variante seleccionada, para construir el dataset final que luego se usa en [modeling.ipynb](modeling.ipynb).

Todas las comparaciones usan el mismo split train/validation, el mismo modelo base y las mismas métricas. De esta forma, si cambia el desempeño, el cambio se puede atribuir a las features agregadas o eliminadas.

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Loading Data
</div>

Se parte de los datasets generados en `preprocessing.ipynb`. En esta etapa todavía usamos los splits pre-one-hot, porque algunas variantes necesitan crear nuevas columnas antes de codificar las variables categóricas.

In [ ]:
SPLITTED_DATA_DIR = "../data/processed/splitted"

X_train = pd.read_csv(f"{SPLITTED_DATA_DIR}/X_train.csv")
X_val = pd.read_csv(f"{SPLITTED_DATA_DIR}/X_val.csv")
y_train = pd.read_csv(f"{SPLITTED_DATA_DIR}/y_train.csv").squeeze()
y_val = pd.read_csv(f"{SPLITTED_DATA_DIR}/y_val.csv").squeeze()

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Experiment Settings
</div>

Estos parámetros definen cómo se construyen las variantes, qué columnas se descartan, qué columnas se codifican y qué marcas se consideran de alta gama.

In [ ]:
REFERENCE_YEAR = 2025
ZERO_KM_THRESHOLD = 100
PREMIUM_BRANDS = const.PREMIUM_BRANDS
BRAND_MODEL_MIN_COUNT = 20

COLUMNS_TO_DROP = [
    "Título",
    "Descripción",
    "Versión"
]

CATEGORICAL_COLS = [
    "Marca",
    "Modelo",
    "Marca_Modelo",
    "Color",
    "Tipo de vendedor",
    "Tipo de combustible",
    "Transmisión"
]

BINARY_MISSING_COLS = ["Con cámara de retroceso"]

<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Candidate Features
</h3>

Se prueban features que intentan capturar información que el modelo puede no estar usando bien con las columnas originales:

- **Uso del vehículo (`usage`)**: agrega `Antigüedad`, `Kilómetros_por_año` y `Es_0km`. Busca representar desgaste y depreciación de forma más directa.
- **Interacción marca-modelo (`brand_model`)**: agrega `Marca_Modelo`, porque el precio suele depender de la combinación completa y no solo de `Marca` o `Modelo` por separado.
- **Marca de alta gama (`premium`)**: agrega `Es_Alta_Gama`, una señal de segmento para marcas premium.
- **Cilindrada faltante (`cilindrada_missing`)**: agrega un indicador para conservar la información de que no se pudo extraer la cilindrada.
- **Recuperación de versión**: transforma la columna `Versión` en variables más simples: un nivel ordinal de equipamiento y señales binarias como `4x4`, `Sport` o `Extra`.

La recuperación de `Versión` es especialmente exploratoria: puede aportar información sobre gama y equipamiento, pero también puede introducir ruido si los patrones no son suficientemente precisos.

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Feature Representation
</div>

Antes de comparar variantes, definimos cómo entra cada tipo de variable al modelo.

- **Features numéricas**
  
  Se incorporan directamente al dataset:
  
  - `Antigüedad`
  - `Kilómetros_por_año`
  - `Version_Tier`

  Estas variables representan cantidades u órdenes numéricos. En particular, `Version_Tier` se interpreta como una escala ordinal de equipamiento, donde valores más altos indican versiones más completas o premium.

- **Features binarias**
  
  Se incorporan como columnas de 0/1:
  
  - `Es_0km`
  - `Es_Alta_Gama`
  - `Cilindrada_missing`
  - `Version_Tier_Unknown`
  - `Version_4x4`
  - `Version_Sport`
  - `Version_Extra`

  Estas columnas indican presencia o ausencia de una condición. No requieren one-hot encoding porque ya están en formato numérico binario.

- **Features categóricas**
  
  Se codifica con **one-hot encoding**:
  
  - `Marca_Modelo`

  El uso de one-hot evita imponer un orden artificial entre categorías nominales. Además, aprender las categorías solo en train evita incorporar información de validation durante la preparación de datos.

<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Version Recovery Patterns
</h3>

Para recuperar información de `Versión`, se usan patrones de texto. Primero se define un tier ordinal:

- `0`: versión base
- `1`: versión media o estándar
- `2`: versión alta
- `3`: versión premium o tope de gama

Además, se crean indicadores binarios para señales específicas que pueden afectar el precio sin representar necesariamente un nivel de equipamiento.

In [ ]:
PREMIUM_VERSION_PATTERNS = [
    r"\bpremium\b",
    r"\bplatinum\b",
    r"\bsummit\b",
    r"\boverland\b",
    r"\bhigh country\b",
    r"\bhigh altitude\b",
    r"\bluxury\b",
    r"\btouring\b",
    r"\bamg\b",
    r"\bm sport\b",
    r"\bm performance\b",
    r"\bs-?line\b",
    r"\br-?line\b",
    r"\braptor\b",
    r"\bsahara\b",
    r"\brubicon\b",
]

HIGH_VERSION_PATTERNS = [
    r"\blimited\b",
    r"\bhighline\b",
    r"\btitanium\b",
    r"\bpremier\b",
    r"\bexclusive\b",
    r"\bfeline\b",
    r"\bshine\b",
    r"\btrailhawk\b",
    r"\bsrx\b",
    r"\bseg\b",
    r"\bxle\b",
    r"\bex-?l\b",
    r"\biconic\b",
    r"\bintens\b",
    r"\bimpetus\b",
    r"\baudace\b",
    r"\bxline\b",
    r"\bsx\b",
    r"\bltz\+?\b",
    r"\bserie[- ]?s\b",
]

MID_VERSION_PATTERNS = [
    r"\bcomfortline\b",
    r"\ballure\b",
    r"\badvance\b",
    r"\bxei\b",
    r"\bxls\b",
    r"\bsrv\b",
    r"\bex\b",
    r"\bgls\b",
    r"\bsel\b",
    r"\blt\b",
    r"\blongitude\b",
    r"\bprivilege\b",
    r"\bdynamique\b",
    r"\bzen\b",
    r"\bfeel\b",
    r"\bstyle\b",
    r"\bfreestyle\b",
    r"\blife\b",
    r"\bse\b",
    r"\bdrive\b",
]

BASE_VERSION_PATTERNS = [
    r"\bbase\b",
    r"\bentry\b",
    r"\btrendline\b",
    r"\bsense\b",
    r"\bactive\b",
    r"\bxli\b",
    r"\bls\b",
    r"\blx\b",
    r"\bxl\b",
    r"\b1lt\b",
    r"\bexpression\b",
    r"\bgl\b",
]

VERSION_4X4_PATTERNS = [
    r"\b4x4\b",
    r"\b4wd\b",
    r"\bawd\b",
    r"\b4motion\b",
    r"\bxdrive\b",
    r"\bquattro\b",
    r"\b4matic\b",
]

VERSION_SPORT_PATTERNS = [
    r"\bsport\b",
    r"\bsportline\b",
    r"\bgt\b",
    r"\bgt[- ]?line\b",
    r"\brs\b",
    r"\babarth\b",
    r"\bgr[- ]?s\b",
    r"\btrailhawk\b",
]

VERSION_EXTRA_PATTERNS = [
    r"\bplus\b",
    r"\bpack\b",
    r"\bpk\b",
    r"\bfull\b",
    r"\btop\b",
    r"\btecho\b",
    r"\bcuero\b",
    r"\bnav\b",
    r"\bbitono\b",
    r"\bbi tono\b",
]

VERSION_TIER_PATTERNS = {
    3: PREMIUM_VERSION_PATTERNS,
    2: HIGH_VERSION_PATTERNS,
    1: MID_VERSION_PATTERNS,
    0: BASE_VERSION_PATTERNS,
}

VERSION_BINARY_PATTERNS = {
    "4x4": VERSION_4X4_PATTERNS,
    "Sport": VERSION_SPORT_PATTERNS,
    "Extra": VERSION_EXTRA_PATTERNS,
}

<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Feature Variant Comparison
</h3>

Se evalúan distintas combinaciones de features con la función `fe.evaluate_feature_variants`. Esta función construye cada variante, aplica one-hot encoding aprendido en train, entrena el modelo y devuelve una tabla comparable de métricas.

La comparación incluye variantes simples y combinadas. No necesariamente gana la que tiene más columnas: una feature puede agregar ruido, redundancia o demasiada dimensionalidad.

In [ ]:
feature_variants = [
    {
        "name": "baseline",
        "feature_blocks": [],
    },
    {
        "name": "usage",
        "feature_blocks": ["usage"],
    },
    {
        "name": "brand_model",
        "feature_blocks": ["brand_model"],
    },
    {
        "name": "premium_brand",
        "feature_blocks": ["premium"],
    },
    {
        "name": "cilindrada_missing",
        "feature_blocks": ["cilindrada_missing"],
    },
    {
        "name": "version_recovery",
        "feature_blocks": ["version"],
    },
    {
        "name": "premium_plus_version",
        "feature_blocks": ["premium", "version"],
    },
    {
        "name": "all_basic_features",
        "feature_blocks": "all",
    },
    {
        "name": "all_basic_features_plus_version",
        "feature_blocks": ["usage", "brand_model", "premium", "cilindrada_missing", "version"],
    },
    {
        "name": "all_basic_without_brand_model_originals",
        "feature_blocks": "all",
        "drop_cols": ["Marca", "Modelo"],
    },
]

In [ ]:
feature_comparison, feature_variant_results = fe.evaluate_feature_variants(
    X_train,
    X_val,
    y_train,
    y_val,
    variants=feature_variants,
    train_func=mod.train_xgboost,
    train_kwargs={"use_log_target": True},
    columns_to_drop=COLUMNS_TO_DROP,
    categorical_cols=CATEGORICAL_COLS,
    binary_missing_cols=BINARY_MISSING_COLS,
    reference_year=REFERENCE_YEAR,
    zero_km_threshold=ZERO_KM_THRESHOLD,
    premium_brands=PREMIUM_BRANDS,
    brand_model_min_count=BRAND_MODEL_MIN_COUNT,
    version_tier_patterns=VERSION_TIER_PATTERNS,
    version_binary_patterns=VERSION_BINARY_PATTERNS,
)

In [ ]:
fe.display_feature_comparison(feature_comparison)

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Best Feature Variant
</div>

In [ ]:
best_feature_name = feature_comparison.iloc[0]["variant"]
best_feature_result = feature_variant_results[best_feature_name]

fe.display_best_variant_summary(feature_comparison, feature_variant_results)

#### Interpretación para completar

Completar luego de ejecutar la comparación:

- ¿Cuál fue la mejor variante?
- ¿Mejoró respecto del baseline?
- ¿La mejora justifica agregar complejidad?
- ¿La recuperación de `Versión` aportó información o agregó ruido?

Este análisis es el que después justifica qué configuración se copia a `feature_engineering.ipynb`.

<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Encoding Strategy Comparison
</h3>

La comparación anterior responde qué bloques de features conviene agregar usando el esquema base de one-hot encoding. Sin embargo, `Marca`, `Modelo` y `Marca_Modelo` tienen muchas categorías, por lo que también vale la pena evaluar si otra estrategia de codificación mejora el desempeño.

En esta sección se compara:

- **One-hot encoding:** crea una columna binaria por categoría. Es más interpretable y no impone un orden artificial, pero puede generar muchas columnas dispersas.
- **Label encoding:** asigna un número entero a cada categoría aprendido solo en train. Reduce dimensionalidad, pero introduce un orden artificial entre categorías nominales.

Para que la comparación sea justa, se prueban dos escenarios:

- **Encoding solo:** se compara la codificación de `Marca`, `Modelo` y `Marca_Modelo` sin acoplarla a todas las demás features.
- **Encoding con features prometedoras:** se prueba la misma codificación junto con `premium` y `brand_model`, que son bloques relevantes para el problema.

Así se separan dos preguntas: qué features agregar y cómo conviene codificar las categorías de alta cardinalidad.

In [ ]:
encoding_one_hot_experiments = [
    {
        "experiment": "one_hot_brand_model",
        "params_name": "xgb_base",
        "params": const.XGB_BASE_PARAMS,
        "feature_blocks": ["brand_model"],
    },
    {
        "experiment": "one_hot_premium_brand_model",
        "params_name": "xgb_base",
        "params": const.XGB_BASE_PARAMS,
        "feature_blocks": ["premium", "brand_model"],
    },
]

encoding_label_experiments = [
    {
        "experiment": "label_brand_model",
        "params_name": "xgb_base",
        "params": const.XGB_BASE_PARAMS,
        "feature_blocks": ["brand_model"],
        "label_encoded_cols": ["Marca", "Modelo", "Marca_Modelo"],
    },
    {
        "experiment": "label_premium_brand_model",
        "params_name": "xgb_base",
        "params": const.XGB_BASE_PARAMS,
        "feature_blocks": ["premium", "brand_model"],
        "label_encoded_cols": ["Marca", "Modelo", "Marca_Modelo"],
    },
]

In [ ]:
encoding_one_hot_comparison, encoding_one_hot_results = exp.run_one_hot_experiments(X_train, y_train, X_val, y_val,
                                                                                    experiments=encoding_one_hot_experiments,)

encoding_label_comparison, encoding_label_results = exp.run_label_encoding_experiments(X_train, y_train, X_val, y_val, 
                                                                                       experiments=encoding_label_experiments,
)

In [ ]:
encoding_strategy_comparison = (
    pd.concat([encoding_one_hot_comparison, encoding_label_comparison], ignore_index=True)
    .sort_values("val_rmse")
    .reset_index(drop=True)
)

exp.style_experiment_comparison(encoding_strategy_comparison)

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Best Encoding Strategy
</div>

In [ ]:
best_encoding_name, best_encoding_result = exp.select_best_experiment(encoding_strategy_comparison, encoding_one_hot_results, encoding_label_results,)

best_encoding_summary = exp.build_best_experiment_summary(encoding_strategy_comparison, best_encoding_name, best_encoding_result,)

best_encoding_summary.style.hide(axis="index").format(exp.EXPERIMENT_METRIC_FORMAT)

#### Interpretación para completar

Completar luego de ejecutar la comparación:

- ¿Label encoding mejora respecto del one-hot?
- ¿La mejora aparece solo cuando se agrega `Marca_Modelo` o también cuando se combina con `premium`?
- Si mejora poco, ¿conviene mantener one-hot por interpretabilidad?
- Si mejora bastante, ¿conviene llevar esta estrategia a `feature_engineering.ipynb` o dejarla como experimento adicional?

Esta sección no reemplaza la comparación de features anterior: la complementa. Primero se eligen las features útiles y después se evalúa si la estrategia de encoding puede mejorar el resultado.

<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Column Drop Comparison
</h3>

Una vez identificada una variante prometedora, se prueba si algunas columnas originales conviene eliminarlas. Este análisis se conoce como ablation: se parte de una configuración base y se observa qué pasa al quitar información.

En particular, se prueban `Color` y `Con cámara de retroceso`, porque en los análisis previos no mostraron una asociación fuerte con `Precio` y pueden sumar columnas one-hot poco informativas.

In [ ]:
DROP_BASE_FEATURE_BLOCKS = ["premium"]

column_drop_comparison, column_drop_results = fe.evaluate_column_drop_variants(
    X_train,
    X_val,
    y_train,
    y_val,
    train_func=mod.train_xgboost,
    base_name="premium_brand",
    feature_blocks=DROP_BASE_FEATURE_BLOCKS,
    columns_to_test={
        "color": "Color",
        "backup_camera": "Con cámara de retroceso",
    },
    train_kwargs={"use_log_target": True},
    columns_to_drop=COLUMNS_TO_DROP,
    categorical_cols=CATEGORICAL_COLS,
    binary_missing_cols=BINARY_MISSING_COLS,
    reference_year=REFERENCE_YEAR,
    zero_km_threshold=ZERO_KM_THRESHOLD,
    premium_brands=PREMIUM_BRANDS,
    brand_model_min_count=BRAND_MODEL_MIN_COUNT,
    version_tier_patterns=VERSION_TIER_PATTERNS,
    version_binary_patterns=VERSION_BINARY_PATTERNS,
)

fe.display_feature_comparison(column_drop_comparison)

<div style="
    text-align: center;
    background-color: rgba(0, 0, 0, 0.3);
    color: white;
    padding: 10px;
    border-radius: 8px;
    font-weight: bold;
">
Best Drop Variant
</div>

In [ ]:
fe.display_best_variant_summary(column_drop_comparison, column_drop_results)

#### Interpretación para completar

Completar luego de ejecutar la comparación:

- ¿Mejora el modelo al eliminar alguna columna?
- ¿La mejora es suficientemente grande o puede deberse a variación del entrenamiento?
- ¿Qué columnas conviene mantener por interpretabilidad aunque no mejoren demasiado?

Si una variante gana de forma consistente, copiar esa decisión a `feature_engineering.ipynb` mediante:

```python
SELECTED_FEATURE_BLOCKS = [...]
SELECTED_DROP_COLS = [...]
```

Por ejemplo:

```python
SELECTED_FEATURE_BLOCKS = ["premium"]
SELECTED_DROP_COLS = ["Color", "Con cámara de retroceso"]
```

<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Version Feature Diagnostics
</h3>

Esta sección ayuda a revisar si la recuperación de `Versión` tiene cobertura suficiente. Si demasiadas filas quedan como `Version_Tier_Unknown`, la feature puede ser débil o necesitar mejores patrones.

In [ ]:
X_train_version_preview, _ = fe.add_text_pattern_features_to_split(
    X_train,
    X_val,
    source_col="Versión",
    tier_patterns=VERSION_TIER_PATTERNS,
    binary_patterns=VERSION_BINARY_PATTERNS,
    prefix="Version",
    default_tier=1,
)

version_feature_cols = [
    "Version_Tier",
    "Version_Tier_Unknown",
    "Version_4x4",
    "Version_Sport",
    "Version_Extra",
]

X_train_version_preview[version_feature_cols].mean().sort_values(ascending=False).to_frame("train_mean")

<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Final Decision
</h3>

Completar al final del experimento:

- Variante seleccionada:
- Columnas eliminadas:
- Métrica de validation:
- Motivo de la elección:

Esta decisión es la que se traslada al notebook final `feature_engineering.ipynb`.